In [0]:
spark.range(5).show()

In [0]:
from pyspark.sql import SparkSession

print(SparkSession.getActiveSession())

## collect_list()

- aggregate function, a synonym of **array_agg** aggregate function

`collect_list ( [ALL | DISTINCT] expr ) [FILTER ( WHERE cond ) ]`

In [0]:
from pyspark.sql import functions as F

# Sample dataframe
data_sample = [("Finance", "Alice"), ("Finance", "Bob"), ("Finance", "Alice"), ("HR", "Charlie")]
df = spark.createDataFrame(data_sample, ["Department", "Employee"])

df_grouped = df.groupBy("Department").agg(F.collect_list("Employee").alias("Employee_list"))

df_grouped.display()


## Explode

* **What it does:** A **transformation function** that takes an array or map column and creates a new row for each element.
* **Key traits:**
  * Flattens nested array/map structures into individual rows.
  * Drops rows where the array is empty or `NULL` (use `explode_outer` if you want to keep empty/null arrays as rows with a `NULL` value).
  * Only one `explode` function is allowed per `SELECT` clause in SQL/PySpark unless handled via LATERAL VIEW or specific lateral joins.

In [0]:
df_exploded = df_grouped.withColumn("exploded_item", F.explode("Employee_list"))
df_exploded.display()

## pivot()

`df.groupBy("column_to_keep").pivot("column_to_rotate").agg_function("column_to_measure")`

In [0]:
data=spark.read.format('csv').option('inferSchema',True).option('header',True).load("/Volumes/sample_data/sales/bigmart/BigMart Sales.csv")

data_selected = data.select("Item_Type","Outlet_Size","Item_MRP")
# data_selected.display()

data_selected.groupBy("Item_Type").pivot("Outlet_Size").agg(F.avg("Item_MRP")).display()

## when_ohterwise()

- if-else logic
- SQL equivalent: CASE-WHEN

In [0]:
from pyspark.sql.functions import when, col
data.withColumn("Veg-flag",when(col("Item_Type")=="Meat","Non-veg").otherwise("veg")).display()

## Windows Function

In [0]:
from pyspark.sql import Window
import pyspark.sql.functions as F

data = [
    ("Engineering", "Mal", 9500),
    ("Engineering", "Zoe", 9500), # Tied salary
    ("Engineering", "Jayne", 8000),
    ("Engineering", "Kaylee", 7500),
    ("HR", "Inara", 9000),
    ("HR", "Simon", 6000)
]
columns = ["department", "employee", "salary"]

df_hr = spark.createDataFrame(data, columns)

# Spec 1: For descriptive rankings and calculations across the ENTIRE department group
dept_spec = Window.partitionBy("department").orderBy(F.desc("salary"))

# Spec 2: For cumulative running aggregations within the department group
running_total_spec = Window.partitionBy("department").orderBy("salary")\
                .rowsBetween(Window.unboundedPreceding,Window.currentRow)

df_hr_analyzed = df_hr \
    .withColumn("dense_rank", F.dense_rank().over(dept_spec))\
    .withColumn("row_number", F.row_number().over(dept_spec))\
    .withColumn("max_dept_salary", F.max("salary").over(dept_spec))\
    .withColumn("diff_from_top_earner", F.col("max_dept_salary") - F.col("salary"))\
    .withColumn("running_dept_avg",F.avg("salary").over(running_total_spec))\
    .drop("max_dept_salary")

df_hr_analyzed.display()